In [ ]:
# ============================================================
# Colab | Tendências CH4 (S5P) por polígono (distance == 10000)
# Saída: 1 imagem por feição (clipped), com bandas de tendência
# Bandas: Theil-Sen (slope, intercept) + Mann-Kendall (tau, z, p, s, var_s)*
# *OBS: As bandas exatas de MK podem variar conforme a versão do EE.
# ============================================================

import ee
import time
import json
import ee
import os
import sys
import collections
# ================================
# 1. Autenticação
# ================================
try:
    ee.Initialize(project=os.environ.get('EE_PROJECT'))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=os.environ.get('EE_PROJECT'))

In [ ]:
# ======================
# 2) Assets de entrada
# ======================
asset_city = 'projects/ee-rnvuefsppgm/assets/CAP_JOIN2'
asset_pca_city = 'projects/nexgenmap/MapBiomas2/LANDSAT/DEGRADACAO/mosaics-ndfia'

In [ ]:
# ============================================================
# 3) Utilitários e classes
# ============================================================

class calculo_group(object):
    """Apoio para detecção de empates (ties) por matching absoluto."""
    def __init__(self, imgCol):
        self.col_to_matches = imgCol

    def function_matches(self, img):
        def iterador_segundo(jj):
            return img.abs().eq(ee.Image(jj).abs())
        matches = self.col_to_matches.map(lambda kk: iterador_segundo(kk))
        # soma de matches;  >1 indica 'tie'
        return ee.ImageCollection(matches).sum()

# ---------- Classe principal do Kendall sazonal ----------
class kendall_trend_sazonal(object):

    options = {
        'year_start': 1985,
        'calc_pValue': True,
        'asset_output': 'projects/rnvuefsppgm/assets/Luana_RSEI_TRENDS_GAP'
    }

    def __init__(self, imColmun, id_mun, name_mun, nRegions, nversion, list_years):
        self.id_mun = id_mun
        self.name_mun = name_mun
        self.id_version = nversion
        self.list_years = list_years

        self.imColmun = ee.ImageCollection(imColmun)
        self.num_img = self.imColmun.size().getInfo()
        print(f"====== Was loaded {self.num_img} image Collection from municipio PC1 =====")

        self.geomet = ee.Geometry(ee.FeatureCollection(nRegions).geometry())
        # Máscara 1/NaN (qualquer valor >0 vira máscara 1)
        self.mask_geom = ee.FeatureCollection(nRegions).reduceToImage(['id_cod'], ee.Reducer.first()).gt(0)

        print("inicializando o calculo de kendall")

    def getDict_numId(self, lst_windowsYears):
        dictMonthY = {}
        for mmes in range(1, 13):
            lst_idNumY = []
            for iyear in lst_windowsYears:
                difYear = iyear - self.options['year_start']
                numbId_tmp = int(difYear * 12) + int(mmes)
                lst_idNumY.append(numbId_tmp)
            dictMonthY[str(mmes)] = lst_idNumY
        return dictMonthY

    def filterSystem_CalculateKendallIndex(self):

        mydictNumIdYear = self.getDict_numId(self.list_years)

        # ---- IMPORTANTE: iniciar acumuladores como FLOAT para preservar decimais ----
        indice_Kendall = ee.Image.constant(0).toFloat()
        p_value_kendall_mean = ee.Image.constant(0).toFloat()
        z_kendall = ee.Image.constant(0).toFloat()
        kendallVariance_mean = ee.Image.constant(0).toFloat()

        for kk, vlist_mes in mydictNumIdYear.items():
            print(f" *** PROCESSING MONTH {kk} ***")

            img_Col_sazonal = self.imColmun.filter(ee.Filter.inList('numId', vlist_mes))

            # Junta pares (i < j) pela chave numId
            afterFilter = ee.Filter.lessThan(leftField='numId', rightField='numId')
            joined_imgCol = ee.ImageCollection(
                ee.Join.saveAll('after').apply(primary=img_Col_sazonal,
                                               secondary=img_Col_sazonal,
                                               condition=afterFilter)
            )

            # Sinal de Kendall acumulado por par (i,j) para cada 'current'
            # Flatten vira uma lista de images
            kendall_ind = joined_imgCol.map(lambda col_current: self.indicador_mann_kendall(col_current))
            # Somatório preserva float
            kendall_ind = ee.ImageCollection(kendall_ind).sum().toFloat()

            indice_Kendall = indice_Kendall.add(kendall_ind)

            if self.options['calc_pValue']:
                # ======= Empates (ties) =======
                def agrupar_ties(marray):
                    length = marray.arrayLength(0)
                    indices = ee.Image([1]).arrayRepeat(0, length).arrayAccum(0, ee.Reducer.sum()).toArray(1)
                    sorted_arr = marray.arraySort()
                    left = sorted_arr.arraySlice(0, 1)
                    right = sorted_arr.arraySlice(0, 0, -1)
                    mask = left.neq(right).arrayCat(ee.Image(ee.Array([[1]])), 0)
                    runIndices = indices.arrayMask(mask)
                    groupSizes = runIndices.arraySlice(0, 1).subtract(runIndices.arraySlice(0, 0, -1))
                    return groupSizes

                nn = len(vlist_mes)
                def factors(img):
                    # b() é a banda única da imagem 'count' ou 'groupSizes' quando usado com array
                    return ee.Image(img).expression('b() * (b() - 1) * (b() * 2 + 5)').toFloat()

                funct_group_size = calculo_group(img_Col_sazonal)
                groups = img_Col_sazonal.map(lambda ff: funct_group_size.function_matches(ff))
                groupSizes = agrupar_ties(ee.ImageCollection(groups).toArray())

                groupFactors = factors(groupSizes)
                groupFactorSum = groupFactors.arrayReduce(ee.Reducer.sum(), [0]).arrayGet([0, 0]).toFloat()

                count = joined_imgCol.count().toFloat()
                kendallVariance = factors(count).subtract(groupFactorSum).divide(18.0).toFloat()
                kendallVariance_mean = kendallVariance_mean.add(kendallVariance)

                # Estatística Z
                zero = indice_Kendall.eq(0).toFloat()
                pos = indice_Kendall.gt(0).toFloat()
                neg = indice_Kendall.lt(0).toFloat()

                # Ajuste de sinal conforme fórmula padrão
                zeta = zero.multiply(0).add(
                    pos.multiply(indice_Kendall.subtract(1)).divide(kendallVariance.sqrt())
                ).add(
                    neg.multiply(indice_Kendall.add(1)).divide(kendallVariance.sqrt())
                ).toFloat()

                z_kendall = z_kendall.add(zeta)

                # CDF normal ~ erf
                def eeCdf(zz):
                    return ee.Image(0.5).multiply(ee.Image(1).add(ee.Image(zz).divide(ee.Image(2).sqrt()).erf())).toFloat()

                # p-value bicaudal
                pp = ee.Image(1).subtract(eeCdf(zeta.abs())).toFloat()
                p_value_kendall_mean = p_value_kendall_mean.add(pp)

        # Reduções finais (mantendo floats)
        # Ensure the final reductions are done on the accumulated images, not collections of images
        p_value_kendall_mean = p_value_kendall_mean.clip(self.geomet).rename('significant').toFloat()
        z_kendall = z_kendall.clip(self.geomet).rename('Z').toFloat()
        kendallVariance_mean = kendallVariance_mean.clip(self.geomet).rename('S').toFloat()
        ind_kendall_exp = indice_Kendall.clip(self.geomet).rename('kendall').toFloat()


        # Parâmetro de corte (metadado coerente)
        p_value = 0.05

        # Empacotar imagem final com bandas em float
        out_img = ee.Image.cat([p_value_kendall_mean, z_kendall, kendallVariance_mean, ind_kendall_exp]) \
                          .updateMask(self.mask_geom)

        # Metadados
        out_img = out_img.set(
            'version', self.id_version,
            'CD_MUN', self.id_mun,
            'NM_MUN', self.name_mun,
            'interval', f"{self.list_years[0]}_{self.list_years[-1]}",
            'mk_first_band', 'PC1',
            'mk_n_images', self.num_img,
            'mk_p_threshold', str(p_value),
            'mk_description', 'MK (S, Z, significant) da PC1 por município'
        )

        name_im = f"mann_kendall_city_sazonal_p_{str(p_value)[2:]}_{self.id_mun}_{self.list_years[0]}_{self.list_years[-1]}"
        self.exportarImagem(out_img, name_im, self.geomet)

    # ---------- soma de sinais para pares (i,j) ----------
    def indicador_mann_kendall(self, current):
        def signal_function(i_im, j_im):
            # clamp(-1,1) e retorna como float
            return ee.Image(j_im).neq(i_im) \
                    .multiply(ee.Image(j_im).subtract(i_im).clamp(-1, 1)).toFloat()

        # Access the list of images from the 'after' property
        afterImages = ee.List(current.get('after'))

        def compare_sign(img):
            return signal_function(current, ee.Image(img)).unmask(0).toFloat()

        # Somatório direto das imagens de sinal
        return ee.ImageCollection(afterImages.map(compare_sign)).sum().toFloat()

    # ---------- Exportar (com pyramidingPolicy = mean para preservar decimais) ----------
    def exportarImagem(self, imgU, nameAl, geomet):

        IdAsset = self.options['asset_output'] + "/" + nameAl

        # Política por banda: mean (evita qualquer quantização/inteirização)
        pyr = {
            "significant": "mean",
            "Z": "mean",
            "S": "mean",
            "kendall": "mean"
        }

        optExp = {
            'image': imgU.toFloat(),       # garantia extra
            'description': nameAl,
            'assetId': IdAsset,
            'pyramidingPolicy': pyr,
            'region': geomet,
            'scale': 30,
            'maxPixels': 1e13
        }

        task = ee.batch.Export.image.toAsset(**optExp)
        task.start()
        print(f"salvando ... {nameAl} !")

In [ ]:
# ============================================================
# 4) Normalização (0-1) por imagem com casting robusto
# ============================================================
def normalizeAndCastCollection(imgCol, bandName, region):
    """Normaliza uma banda específica em uma coleção (0-1) e força float."""
    imgCol = ee.ImageCollection(imgCol)

    def normalize_and_cast_image(img):
        dct = ee.Dictionary(
            img.select(bandName).reduceRegion(
                reducer=ee.Reducer.minMax(),
                geometry=ee.Geometry(region.geometry() if isinstance(region, ee.FeatureCollection) else region),
                scale=30,
                maxPixels=1e13,
                bestEffort=True
            )
        )

        # Chaves esperadas do minMax
        kmin = f"{bandName}_min"
        kmax = f"{bandName}_max"

        # Se a chave não existir (imagens vazias sobre a região), usa defaults
        min_val = ee.Number(ee.Algorithms.If(dct.contains(kmin), dct.get(kmin), 0))
        max_val = ee.Number(ee.Algorithms.If(dct.contains(kmax), dct.get(kmax), 1))

        value_range = max_val.subtract(min_val)
        safeRange = ee.Number(ee.Algorithms.If(value_range.gt(0), value_range, 1))

        normalized_band = img.select(bandName).subtract(min_val).divide(safeRange).toFloat().rename(bandName)

        return normalized_band.copyProperties(img, img.propertyNames())

    return imgCol.map(normalize_and_cast_image)


In [ ]:
# ============================================================
# 5) Gap-fill forward (paralelizável) preservando tipo float
# ============================================================
def applyGapFillParallel(imgCol):
    """Preenche mascarados propagando valores anteriores (forward-fill)."""
    imgCol = ee.ImageCollection(imgCol).sort('system:time_start')
    size = imgCol.size()
    empty_collection = ee.ImageCollection([])

    def compute_gapfill():
        # Composto com primeiro valor não-nulo ao longo do tempo
        forwardFill = imgCol.reduce(ee.Reducer.firstNonNull())

        firstImage = ee.Image(imgCol.first())
        restOfCol = ee.ImageCollection.fromImages(imgCol.toList(size).slice(1))

        # Preenche a primeira imagem com o composto e força float
        filledFirstImage = firstImage.unmask(forwardFill).toFloat().copyProperties(
            source=firstImage,
            properties=['system:time_start', 'system:index', 'CD_MUN', 'NM_MUN', 'month', 'year', 'numId']
        )

        def accumulate(currentImage, previousList):
            previousList = ee.List(previousList)
            previousImage = ee.Image(previousList.get(-1))
            filledCurrentImage = ee.Image(currentImage).unmask(previousImage).toFloat().copyProperties(
                source=currentImage,
                properties=['system:time_start', 'system:index', 'CD_MUN', 'NM_MUN', 'month', 'year', 'numId']
            )
            return previousList.add(filledCurrentImage)

        initialList = ee.List([filledFirstImage])
        filledList = ee.List(restOfCol.iterate(accumulate, initialList))
        return ee.ImageCollection.fromImages(filledList)

    return ee.ImageCollection(ee.Algorithms.If(size.gt(0), compute_gapfill(), empty_collection))


In [ ]:
# ============================================================
# 6) Helpers
# ============================================================
def set_id_numero(img):
    difYear = ee.Number(img.get('year')).subtract(1985)
    numbId = ee.Number(difYear).multiply(12).add(ee.Number(img.get('month')))
    return img.set('numId', numbId)

# ============================================================
# 7) Parâmetros gerais e execução
# ============================================================
year_init = 1985
year_end = 2025

pca_city = ee.ImageCollection(asset_pca_city).map(lambda im: set_id_numero(im))
band_act = 'PC1'
version = '1'

# Geometrias (municípios)
allGeometries = ee.FeatureCollection(asset_city)
print(f"We loaded {allGeometries.size().getInfo()} Geometries from all big City ")
allGeometries = allGeometries.map(lambda feat: feat.set('id_cod', 1))

# Lista (NM_MUN, CD_MUN)
lst_codSHP = allGeometries.reduceColumns(ee.Reducer.toList(2), ['NM_MUN', 'CD_MUN']).get('list').getInfo()
print("numero de cidades ", len(lst_codSHP))

# Loop principal
for cc, cod_city in enumerate(lst_codSHP[1:]):  # mantém seu offset original
    cd_mun = cod_city[1]
    nm_mun = cod_city[0]
    im_col = pca_city.filter(ee.Filter.eq('NM_MUN', nm_mun)).select(band_act)
    print(f"#{cc}  city >>>  {nm_mun} >>> {im_col.size().getInfo()}")

    my_region = allGeometries.filter(ee.Filter.eq('NM_MUN', nm_mun))

    # anos presentes (distintos)
    lst_year = ee.List(
        im_col.reduceColumns(ee.Reducer.toList(), ['year']).get('list')
    ).distinct().getInfo()

    # Normaliza PC1 para [0,1] em float
    col_pca = normalizeAndCastCollection(im_col, band_act, my_region)

    # Gap-fill forward em float
    collect_pca = applyGapFillParallel(col_pca)

    # Kendall Sazonal
    trend_manKendall_city = kendall_trend_sazonal(
        ee.ImageCollection(collect_pca), cd_mun, nm_mun, my_region, version, lst_year
    )
    trend_manKendall_city.filterSystem_CalculateKendallIndex()